# ЛР6 · Выполненные проверки без LLM

Этот ноутбук проверяет корпус, retrieval и код. Модель здесь не вызывается; accuracy/F1 ответов модели не измерены.

In [1]:
from pathlib import Path
import os
PROJECT=Path.cwd()
if not (PROJECT/'src').exists():
    PROJECT=PROJECT/'ai-detective'
os.chdir(PROJECT)
from src.io import load_data
from validate_data import validate
D,C,Q=load_data()
validate(D,C,Q)

{'documents': 10, 'words': 5194, 'claims': 36, 'questions': 12}

## Покрытие эталонных оснований в BM25 top-15

Это характеристика поиска на данном корпусе, не результат генерации.

In [2]:
import pandas as pd
from src.retrieval import retrieve
rows=[]
for q in Q:
    predicted={c['claim_id'] for c in retrieve(q['question'],C)}
    gold=set(q['gold_evidence_ids']+q['gold_counterevidence_ids'])
    rows.append({'question_id':q['question_id'],'coverage':len(predicted&gold)/len(gold),'missing':','.join(sorted(gold-predicted))})
pd.DataFrame(rows)

,question_id,coverage,missing
0,Q01,1.0,
1,Q02,1.0,
2,Q03,1.0,
3,Q04,1.0,
4,Q05,1.0,
5,Q06,1.0,
6,Q07,1.0,
7,Q08,1.0,
8,Q09,1.0,
9,Q10,1.0,


## Реальные автоматические проверки

LLM endpoint в тесте отказа подменяется тестовым транспортом только внутри pytest. Исследовательских предсказаний он не создаёт.

In [3]:
import subprocess,sys
r=subprocess.run([sys.executable,'-m','pytest','-q'],capture_output=True,text=True)
print(r.stdout)
assert r.returncode==0,r.stderr

............................                                             [100%]
28 passed in 1.35s



## Статус

Проверки данных и кода завершены. Следующий этап — реальный запуск Qwen и совместная проверка содержательных эталонов. Готового отчёта с результатами модели пока нет.